# MongoDB JSONL Loader for Healthcare Test Questions

---

**Copyright © 2026 Skip Snow. All rights reserved.**

**Coding Engineer:** Claude Sonnet 4.5 (Anthropic)

**Date:** January 25, 2026

---

This notebook provides functionality to load JSONL test questions on doctor quantities into MongoDB collections.

In [ ]:
import json
from typing import Dict, List, Any
from datetime import datetime
from Utilities.ChatHealthyMongoUtilities import ChatHealthyMongoUtilities
from pymongo import MongoClient

In [ ]:
def loadTestQuestionsOnDocQuantities(
    file_path: str,
    mongo_connection,
    database_name: str,
    collection_name: str
) -> List[Dict[str, Any]]:
    """
    Load test questions from JSONL file into MongoDB collection.
    
    If the collection exists, it is dropped before loading new data to ensure a clean state.
    Each line in the JSONL file becomes one document in the MongoDB collection.
    
    Args:
        file_path (str): Path to the JSONL file containing test questions
        mongo_connection: MongoDB client connection object (pymongo.MongoClient)
        database_name (str): Name of the database to use
        collection_name (str): Name of the collection to create/overwrite
    
    Returns:
        List[Dict[str, Any]]: List of statistics dictionaries containing:
            - operation: str, type of operation performed
            - database: str, database name
            - collection: str, collection name
            - file_path: str, source file path
            - records_read: int, number of records read from file
            - records_inserted: int, number of records inserted into MongoDB
            - records_with_valid_codes: int, records with non-Unknown specialty codes
            - records_with_unknown_codes: int, records with Unknown specialty codes
            - success_rate: float, percentage of records with valid codes
            - collection_dropped: bool, whether existing collection was dropped
            - timestamp: str, ISO timestamp of operation
            - skipped_lines: int, number of lines skipped due to JSON errors
    
    Raises:
        FileNotFoundError: If the specified JSONL file does not exist
        ValueError: If file contains no valid JSON records
        ConnectionError: If MongoDB connection fails or is invalid
        RuntimeError: If MongoDB operations fail during execution
    
    Example:
        >>> from pymongo import MongoClient
        >>> client = MongoClient('mongodb://localhost:27017/')
        >>> stats = loadTestQuestionsOnDocQuantities(
        ...     'processed_physicians_phd_only.jsonl',
        ...     client,
        ...     'healthcare_db',
        ...     'test_questions'
        ... )
        >>> print(stats[0]['records_inserted'])
    """
    
    # Validate MongoDB connection
    try:
        mongo_connection.server_info()
    except Exception as e:
        raise ConnectionError(
            f"Failed to connect to MongoDB. Ensure MongoDB is running and accessible. "
            f"Original error: {type(e).__name__}: {str(e)}"
        )
    
    # Validate file exists
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            pass
    except FileNotFoundError:
        raise FileNotFoundError(
            f"JSONL file not found at path: '{file_path}'. "
            f"Please verify the file path and ensure the file exists."
        )
    except PermissionError:
        raise PermissionError(
            f"Permission denied when accessing file: '{file_path}'. "
            f"Please check file permissions."
        )
    except Exception as e:
        raise RuntimeError(
            f"Unexpected error accessing file '{file_path}': {type(e).__name__}: {str(e)}"
        )
    
    # Access database
    try:
        db = mongo_connection[database_name]
    except Exception as e:
        raise RuntimeError(
            f"Failed to access database '{database_name}': {type(e).__name__}: {str(e)}"
        )
    
    # Check if collection exists and drop it
    collection_dropped = False
    try:
        if collection_name in db.list_collection_names():
            db[collection_name].drop()
            collection_dropped = True
    except Exception as e:
        raise RuntimeError(
            f"Failed to drop existing collection '{collection_name}': "
            f"{type(e).__name__}: {str(e)}"
        )
    
    # Get collection reference
    try:
        collection = db[collection_name]
    except Exception as e:
        raise RuntimeError(
            f"Failed to create collection '{collection_name}': {type(e).__name__}: {str(e)}"
        )
    
    # Read JSONL file
    records = []
    skipped_lines = 0
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                try:
                    record = json.loads(line.strip())
                    records.append(record)
                except json.JSONDecodeError:
                    skipped_lines += 1
    except Exception as e:
        raise RuntimeError(
            f"Error reading JSONL file '{file_path}': {type(e).__name__}: {str(e)}"
        )
    
    # Validate we have records
    if not records:
        raise ValueError(
            f"No valid JSON records found in file '{file_path}'. "
            f"The file may be empty or contain only invalid JSON. "
            f"Skipped lines: {skipped_lines}"
        )
    
    # Insert records into MongoDB
    try:
        result = collection.insert_many(records)
        records_inserted = len(result.inserted_ids)
    except Exception as e:
        raise RuntimeError(
            f"Failed to insert records into MongoDB collection '{collection_name}': "
            f"{type(e).__name__}: {str(e)}"
        )
    
    # Calculate statistics
    try:
        records_with_valid_codes = collection.count_documents({
            'Answer.Codes': {'$exists': True, '$ne': ['Unknown']}
        })
        
        records_with_unknown_codes = collection.count_documents({
            'Answer.Codes': ['Unknown']
        })
        
        success_rate = (records_with_valid_codes / records_inserted * 100) if records_inserted > 0 else 0.0
        
    except Exception as e:
        raise RuntimeError(
            f"Failed to calculate statistics for collection '{collection_name}': "
            f"{type(e).__name__}: {str(e)}"
        )
    
    # Return statistics as list of dictionaries
    statistics = [
        {
            'operation': 'load_jsonl_to_mongodb',
            'database': database_name,
            'collection': collection_name,
            'file_path': file_path,
            'records_read': len(records),
            'records_inserted': records_inserted,
            'records_with_valid_codes': records_with_valid_codes,
            'records_with_unknown_codes': records_with_unknown_codes,
            'success_rate': round(success_rate, 2),
            'collection_dropped': collection_dropped,
            'timestamp': datetime.utcnow().isoformat() + 'Z',
            'skipped_lines': skipped_lines
        }
    ]
    
    return statistics

## Usage Example

```python


# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')

# Load data and get statistics
stats = loadTestQuestionsOnDocQuantities(
    file_path='processed_physicians_phd_only.jsonl',
    mongo_connection=client,
    database_name='healthcare_db',
    collection_name='test_questions'
)

# Display statistics
for stat in stats:
    print(f"Records inserted: {stat['records_inserted']}")
    print(f"Success rate: {stat['success_rate']}%")
    print(f"Valid codes: {stat['records_with_valid_codes']}")
    print(f"Unknown codes: {stat['records_with_unknown_codes']}")
```